# 02e — High-resolution retrain (yolov8s @ 960)

Targets the two measured weaknesses of `baseline_yolov8s` (30 ep @ 640, test mAP50 0.2472 @ conf=0.25):

- **Small objects** — recall@0.5 was 0.405 small vs 0.736 large, and 24,104 of 30,832 test boxes are small. Training and inferring at 960 instead of 640 gives every box ~2.25x the pixels, which is the single biggest lever available without new data.
- **The class tail** — 46 of 102 evaluable classes at AP=0. `copy_paste` augmentation pastes object instances across images, which disproportionately helps rare classes and small objects.

Budget guardrails, since this must fit one Colab T4 session:

- `time=5.5` hard-caps training at 5.5 wall-clock hours no matter what `epochs` says; Ultralytics stops on schedule and keeps `best.pt`.
- Warm-started from `baseline_yolov8s/weights/best.pt` (already knows the 148 classes) instead of COCO weights, so the run spends its limited hours adapting to 960px rather than relearning the taxonomy.
- Runs write directly to `RUNS_DIR` on Drive, so a dead session loses nothing; re-running the train cell auto-resumes from `last.pt`.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install ultralytics -q


In [ ]:
import sys
from pathlib import Path

SCRIPTS_DIR = Path("/content/drive/Shareddrives/Computer Vision Final Project/Modeling/YOLO")
sys.path.insert(0, str(SCRIPTS_DIR))

from yolo_common_colab import *

# Extracts the dataset tar to local disk on first call (~5-25 min depending on Drive throughput).
data_yaml_path, data_yaml, unified_classes, train_image_paths, val_image_paths, test_image_paths = load_dataset()
print(f"{len(train_image_paths)} train / {len(val_image_paths)} val / {len(test_image_paths)} test")


In [ ]:
from ultralytics import YOLO

RUN_NAME = "highres_yolov8s_960"
run_dir = RUNS_DIR / RUN_NAME
last_ckpt = run_dir / "weights" / "last.pt"
baseline_ckpt = RUNS_DIR / "baseline_yolov8s" / "weights" / "best.pt"

# If a previous session died mid-run, pick up where it left off; otherwise warm-start
# from the 640px baseline so the limited T4 hours go to adapting, not relearning.
resuming = last_ckpt.exists()
model = YOLO(str(last_ckpt if resuming else baseline_ckpt))
print(f"{'Resuming' if resuming else 'Warm-starting'} from {last_ckpt if resuming else baseline_ckpt}")

results = model.train(
    data=str(data_yaml_path),
    project=str(RUNS_DIR),
    name=RUN_NAME,
    exist_ok=True,
    resume=resuming,

    epochs=60,          # upper bound; the time cap below is what actually ends the run
    time=5.5,           # hard wall-clock cap in hours -- leaves ~2h of the session for eval
    patience=15,

    imgsz=960,
    batch=-1,           # auto-fit the T4's 15GB (expect ~8-12 at 960 for v8s)

    copy_paste=0.3,     # instance paste-in: the tail-class / small-object lever
    mixup=0.1,
    close_mosaic=10,    # last 10 epochs on clean images, as in the baseline recipe

    seed=42,
    plots=True,
)


## Test-set evaluation

Two numbers on purpose:

1. **`conf=0.25` protocol** — directly comparable to the 0.2472 / 0.1461 the baseline reported in `02d` and `MODEL_SELECTION.md`. This is the swap-in decision number.
2. **`conf=0.001` protocol** — standard COCO-style scoring (the baseline's 640px checkpoint was never re-scored this way on test; its val-split number was 0.366). Report this as the honest headline metric for both models.


In [ ]:
best = YOLO(str(run_dir / "weights" / "best.pt"))

print("=== conf=0.25 (baseline-comparable protocol) ===")
m = best.val(data=str(data_yaml_path), split="test", imgsz=960, conf=0.25, iou=0.45)
print(f"mAP@50: {m.box.map50:.4f}   mAP@50:95: {m.box.map:.4f}   P: {m.box.mp:.4f}   R: {m.box.mr:.4f}")

print("\n=== conf=0.001 (standard COCO-style protocol) ===")
m2 = best.val(data=str(data_yaml_path), split="test", imgsz=960, conf=0.001, iou=0.45)
print(f"mAP@50: {m2.box.map50:.4f}   mAP@50:95: {m2.box.map:.4f}")

# Decision rule: swap into the app if the conf=0.25 numbers beat 0.2472 / 0.1461.


## Export for the app

Only run after the eval above beats the baseline. The app loads `best.pt` via Ultralytics directly, so copying that file down is sufficient; the ONNX export is for parity with the baseline's deployment artifact. Note `imgsz=960` — the app's detector must infer at the resolution the model was trained at.


In [ ]:
export_path = best.export(format="onnx", imgsz=960, dynamic=True)
print(f"Exported: {export_path}")
print(f"Weights to download for the app: {run_dir / 'weights' / 'best.pt'}")
